# Hard-wall quenched log-polar effective Hamiltonian

This local CPU notebook reconstructs the endpoint-identifiable left polar generator from the 300 completed hard-wall maximally mixed trajectories at $N_x=20$ and $N_y=20,30,40$. It evaluates the matrix logarithm trajectory by trajectory, restores translation invariance by a $y$ twirl, and only then performs the quenched sample average. It compares the half-filled ground states of the quenched and typical effective Hamiltonians with the actual endpoint steady states.

In [ ]:
# Runtime and complete editable analysis configuration. Run this cell first.
import os
from pathlib import Path

CONSTRUCTION = 'hard'
CPU_RANGE = ''                 # Example: '0-15' or '0-7,16-23'; blank keeps current affinity.
N_WORKERS = 1                 # Parallel trajectories within each five-sample source shard.
NY_VALUES = (20, 30, 40)
MAX_SAMPLES_PER_NY = None     # Set to 1 for a smoke run; None analyzes all 100 per size.
VERIFY_SOURCE_HASHES = True
POLAR_LOG_CAPS = (8.0, 10.0, 12.0)
DEFAULT_POLAR_LOG_CAP = 10.0
RENYI_ORDERS = (1, 2, 3)
BOUNDARY_TWISTS = (-1.0e-7, 1.0e-7)
BOOTSTRAP_REPLICATES = 2000
JACKKNIFE_GROUPS = 10
OUTPUT_REVISION = 'hard_quenched_log_polar_v1'
OUTPUT_ROOT_OVERRIDE = ''       # Absolute or repo-relative path; blank uses this bundle.

def _parse_cpu_range(spec):
    selected = set()
    for token in spec.replace(' ', '').split(','):
        if not token:
            continue
        if '-' in token:
            lo, hi = (int(value) for value in token.split('-', 1))
            selected.update(range(lo, hi + 1))
        else:
            selected.add(int(token))
    return selected

available_cpus = set(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else set()
if CPU_RANGE and available_cpus:
    requested_cpus = _parse_cpu_range(CPU_RANGE)
    if not requested_cpus or not requested_cpus.issubset(available_cpus):
        raise ValueError(f'CPU_RANGE={CPU_RANGE!r} is not contained in available CPUs {sorted(available_cpus)}')
    os.sched_setaffinity(0, requested_cpus)
allocated_cpus = sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else []
blas_threads = 1 if N_WORKERS > 1 else max(1, len(allocated_cpus))
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = str(blas_threads)
print({'allocated_cpus': allocated_cpus, 'workers': N_WORKERS, 'blas_threads': blas_threads})

## Theory and estimator order

For one conditioned record $\xi$, maximally mixed input gives $\hat\rho_\xi\propto\hat V_\xi\hat V_\xi^\dagger$. With the endpoint-identifiable left polar decomposition $t_\xi=P_{L,\xi}u_\xi$, the active-slab centered covariance obeys

$$A_\xi\equiv\log P_{L,\xi}=\operatorname{arctanh}G_\xi,\qquad h_\xi=-2A_\xi.$$

The required quenched order is $G_\xi\to A_\xi\to\mathcal T_y(A_\xi)\to\overline{A}_{\rm q}$. The annealed construction $\operatorname{arctanh}[\mathcal T_y(\overline{G})]$ is retained only as a control. Because the endpoints are nearly pure, $A_\xi$ is capped spectrally at $\Lambda=8,10,12$; only cap-stable conclusions are released. Entropy establishes the Cardy--Calabrese central charge, while oppositely directed wall branches establish chirality.

In [ ]:
import json
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

candidate = Path.cwd()
if not (candidate / 'polar_effective_hamiltonian.py').is_file():
    candidate = Path('00_WORKSPACE/CURRENT/final_production_new_designs/07_maxmix_hard_soft_purification').resolve()
BUNDLE_ROOT = candidate
if not (BUNDLE_ROOT / 'polar_effective_hamiltonian.py').is_file():
    raise FileNotFoundError('Could not locate the campaign-07 analysis bundle')
sys.path.insert(0, str(BUNDLE_ROOT))
import polar_effective_hamiltonian as polar

SPEC = polar.HARD_SPEC
DATA_ROOT = SPEC.data_root
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve() if OUTPUT_ROOT_OVERRIDE else BUNDLE_ROOT / 'analysis_outputs' / OUTPUT_REVISION
CACHE_ROOT = OUTPUT_ROOT / 'sample_cache'
FIGURE_ROOT = OUTPUT_ROOT / 'figure_assets'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = polar.AnalysisConfig(
    construction=CONSTRUCTION, caps=POLAR_LOG_CAPS, default_cap=DEFAULT_POLAR_LOG_CAP,
    renyi_orders=RENYI_ORDERS, twists=BOUNDARY_TWISTS,
    bootstrap_replicates=BOOTSTRAP_REPLICATES, jackknife_groups=JACKKNIFE_GROUPS,
)
mpl.rcParams.update({'font.family': 'CMU Sans Serif', 'font.size': 8, 'axes.linewidth': 0.8})
print(json.dumps({
    'bundle_root': str(BUNDLE_ROOT), 'data_root': str(DATA_ROOT),
    'output_root': str(OUTPUT_ROOT), 'analysis_config': CONFIG.identity(),
}, indent=2))

## Exact B0 estimator calibration

Before touching the stochastic endpoints, reproduce the repository's accepted hard-exterior Cardy--Calabrese calibration. This validates the two-wall factor and the $8\leq A_y\leq N_y/2$ fit convention, not the adaptive-circuit result.

In [ ]:
B0_TABLE = BUNDLE_ROOT.parents[1] / 'experiment_review' / 'b0_exact_domain_wall' / 'results' / '20260816_183928' / 'processed' / 'tables' / 'entropy_fits.csv'
b0 = pd.read_csv(B0_TABLE)
b0_check = b0[(b0.construction == 'hard_exterior') & (b0.geometry_key == 'hard_exterior__Nx020__Ny048') & (b0.quantity == 'full_strip')].copy()
if set(b0_check.q) != {1, 2, 3} or np.max(np.abs(b0_check.c_estimate - 1.0)) >= 0.02:
    raise RuntimeError('Exact B0 Cardy--Calabrese calibration did not reproduce c_w=1')
display(b0_check[['q', 'slope', 'c_estimate', 'r2', 'ay_min', 'ay_max']])

## Verify the completed campaign inventory

The input is immutable hard-v2 data. Completion metadata, source identity, byte counts, optional SHA-256 readback, sample coverage, and the active-slab transfer dimension are verified before any cache is reused or created.

In [ ]:
records = polar.discover_samples(
    DATA_ROOT, construction=CONSTRUCTION, ny_values=NY_VALUES, verify_hashes=VERIFY_SOURCE_HASHES, require_complete=True
)
inventory = pd.DataFrame([{'Ny': row.ny, 'sample': row.sample_index, 'source': row.result_path.name} for row in records])
inventory_summary = inventory.groupby('Ny').agg(samples=('sample', 'count'), first=('sample', 'min'), last=('sample', 'max'))
display(inventory_summary)
campaign_parameters = {
    'sampling_revision': SPEC.revision, 'configuration_hash': SPEC.configuration_hash,
    'construction': SPEC.construction, 'Nx': polar.EXPECTED_NX, 'Ny': list(NY_VALUES),
    'cycles': {str(ny): 4 * ny for ny in NY_VALUES}, 'dtype': 'complex128',
    'active_x': list(SPEC.active_x), 'active_dimension': {str(ny): SPEC.transfer_mode_count(ny) for ny in NY_VALUES},
    'trajectory_count': len(records),
}
print(json.dumps(campaign_parameters, indent=2))
print(f'verified {len(records)} trajectory records; source hashes={VERIFY_SOURCE_HASHES}')

## Build or resume trajectory caches

Each cache contains only the $y$-twirled active-slab kernels, endpoint and half-filled entropy curves, cap diagnostics, and numerical residuals. A checksum-bound completion JSON is written last. Restarting the notebook skips verified caches.

In [ ]:
cache_report = polar.build_sample_caches(
    records, cache_root=CACHE_ROOT, config=CONFIG, progress=tqdm,
    max_samples_per_ny=MAX_SAMPLES_PER_NY, workers=N_WORKERS,
)
print('[cache completion]', json.dumps(cache_report, indent=2))

## Aggregate quenched, typical, annealed, and flattened analyses

The quenched mean averages the already-logged and twirled operators. Typical-sample statistics retain trajectories as the independent units. The aggregate stage writes machine-readable spectra, entropy curves, fits, numerical diagnostics, compact plotting arrays, and the claim ledger.

In [ ]:
analysis = polar.aggregate_analysis(
    records, cache_root=CACHE_ROOT, output_root=OUTPUT_ROOT, config=CONFIG,
    max_samples_per_ny=MAX_SAMPLES_PER_NY,
)
summary = analysis['summary']
spectrum_df = pd.read_csv(OUTPUT_ROOT / 'spectrum_rows.csv')
velocity_df = pd.read_csv(OUTPUT_ROOT / 'velocity_summary.csv')
entropy_df = pd.read_csv(OUTPUT_ROOT / 'entropy_curves.csv')
fit_df = pd.read_csv(OUTPUT_ROOT / 'entropy_fit_summary.csv')
diagnostics_df = pd.read_csv(OUTPUT_ROOT / 'numerical_diagnostics.csv')
arrays = np.load(OUTPUT_ROOT / 'notebook_arrays.npz', allow_pickle=False)
print(json.dumps(summary['claim_ledger'], indent=2))

## Quenched and typical wall spectra

Gray points are the six quenched-mean modes nearest zero. Colored solid branches are localized on the two walls; dashed branches are trajectory-wise means. The ordinate is the endpoint Hamiltonian $h=-2\log P_L$.

In [ ]:
colors = ('#d62728', '#1f77b4')
markers = ('^', 'o')
fig, axes = plt.subplots(2, len(NY_VALUES), figsize=(7.05, 4.35), sharex='col')
for panel, ny in enumerate(NY_VALUES):
    key = f'Ny{ny:03d}_cap{DEFAULT_POLAR_LOG_CAP:g}'
    ky = arrays[f'{key}_quenched_ky']
    low = arrays[f'{key}_quenched_low_energy']
    qwall = arrays[f'{key}_quenched_wall_energy']
    twall = arrays[f'{key}_typical_wall_energy_mean']
    for row, scale in enumerate((1.0, 1.0 / (4 * ny))):
        ax = axes[row, panel]
        for band in range(low.shape[1]):
            ax.plot(ky, scale * low[:, band], '.', ms=2.0, color='0.72', zorder=1)
        for wall in range(2):
            ax.plot(ky, scale * qwall[:, wall], color=colors[wall], marker=markers[wall], ms=2.7, lw=1.0, label=f'wall {wall+1}, quenched')
            ax.plot(ky, scale * twall[:, wall], color=colors[wall], ls='--', lw=0.9, alpha=0.9, label=f'wall {wall+1}, typical')
        ax.axhline(0, color='black', ls=':', lw=0.8)
        ax.tick_params(direction='in', top=True, right=True)
        ax.text(-0.18, 1.03, f'({chr(97 + row * len(NY_VALUES) + panel)})', transform=ax.transAxes, fontsize=9)
    axes[0, panel].set_title(rf'$N_y={ny}$')
    axes[1, panel].set_xlabel(r'$k_y$')
axes[0, 0].set_ylabel(r'$h_{\mathrm{eff}}=-2\log P_L$')
axes[1, 0].set_ylabel(r'$h_{\mathrm{eff}}/(4N_y)$')
axes[0, -1].legend(frameon=False, fontsize=5.5, ncol=1, loc='best')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'quenched_typical_wall_spectrum.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'quenched_typical_wall_spectrum.png', dpi=300, bbox_inches='tight')
plt.show()

## Localized branch velocities and cap stability

Opposite velocity signs on the two spatially separated walls are the chirality diagnostic. This panel exposes the cap dependence rather than hiding it.

In [ ]:
fig, axes = plt.subplots(1, len(NY_VALUES), figsize=(7.05, 2.35), sharey=True)
for panel, (ax, ny) in enumerate(zip(np.atleast_1d(axes), NY_VALUES)):
    for ci, cap in enumerate(POLAR_LOG_CAPS):
        velocity = arrays[f'Ny{ny:03d}_cap{cap:g}_quenched_velocity']
        for wall in range(2):
            ax.plot(cap + (wall - 0.5) * 0.10, velocity[wall], marker=markers[wall], color=colors[wall], ms=5, ls='none', label=f'wall {wall+1}' if ci == 0 else None)
    ax.axhline(0, color='black', ls='--', lw=0.8)
    ax.set_title(rf'$N_y={ny}$')
    ax.set_xlabel(r'polar cap $\Lambda$')
    ax.tick_params(direction='in', top=True, right=True)
    ax.text(-0.18, 1.03, f'({chr(97+panel)})', transform=ax.transAxes, fontsize=9)
axes[0].set_ylabel(r'$\partial h_{\mathrm{eff}}/\partial k_y$')
axes[-1].legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'wall_velocity_cap_sensitivity.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'wall_velocity_cap_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

## Cardy--Calabrese entropy curves

The actual endpoint curve averages periodic origins inside each trajectory. The typical effective-Hamiltonian curve is formed trajectory by trajectory. The quenched curve is the half-filled ground state of the sample-averaged log-polar Hamiltonian. Error bars are whole-trajectory 95% intervals where applicable.

In [ ]:
family_style = {
    'actual_endpoint': ('#1f77b4', 'o', '-', 'actual endpoint'),
    'typical_sample': ('#2ca02c', 's', '--', r'typical $h_\xi$ ground state'),
    'quenched_mean': ('#d62728', '^', ':', r'quenched $\overline{h}_q$ ground state'),
}
fig, axes = plt.subplots(1, len(NY_VALUES), figsize=(7.05, 2.55), sharey=False)
for panel, (ax, ny) in enumerate(zip(np.atleast_1d(axes), NY_VALUES)):
    for family, (color, marker, linestyle, label) in family_style.items():
        sub = entropy_df[(entropy_df.family == family) & (entropy_df.Ny == ny) & (entropy_df.renyi_order == 1)]
        if family != 'actual_endpoint':
            sub = sub[np.isclose(sub.cap.astype(float), DEFAULT_POLAR_LOG_CAP) & np.isclose(sub.twist.astype(float), max(BOUNDARY_TWISTS))]
        sub = sub.sort_values('Ay')
        x = polar.log_chord(ny, sub.Ay.to_numpy())
        ax.plot(x, sub['mean'], color=color, marker=marker, ms=3, lw=1.0, ls=linestyle, label=label)
        if family != 'quenched_mean':
            ax.fill_between(x, sub.ci_low, sub.ci_high, color=color, alpha=0.12, lw=0)
        fit_mask = sub.Ay.to_numpy() >= CONFIG.fit_ay_min
        if fit_mask.sum() >= 3:
            coefficient = np.polyfit(x[fit_mask], sub['mean'].to_numpy()[fit_mask], 1)
            ax.plot(x[fit_mask], np.polyval(coefficient, x[fit_mask]), color=color, lw=0.8, alpha=0.8)
    ax.set_title(rf'$N_y={ny}$')
    ax.set_xlabel(r'$\log[(N_y/\pi)\sin(\pi A_y/N_y)]$')
    ax.tick_params(direction='in', top=True, right=True)
    ax.text(-0.18, 1.03, f'({chr(97+panel)})', transform=ax.transAxes, fontsize=9)
axes[0].set_ylabel(r'$S_1(A_y)$')
axes[-1].legend(frameon=False, fontsize=6)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'cardy_calabrese_entropy_curves.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'cardy_calabrese_entropy_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## Extracted central charges

The two-wall conversion is $c_{w,n}=6a_n/(1+1/n)$. Agreement among $n=1,2,3$ is stronger than one successful von Neumann fit.

In [ ]:
fig, axes = plt.subplots(1, len(RENYI_ORDERS), figsize=(7.05, 2.45), sharey=True)
for panel, (ax, order) in enumerate(zip(np.atleast_1d(axes), RENYI_ORDERS)):
    for family, (color, marker, linestyle, label) in family_style.items():
        sub = fit_df[(fit_df.family == family) & (fit_df.renyi_order == order) & (fit_df.fit_variant == 'primary')]
        if family != 'actual_endpoint':
            sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
        sub = sub.sort_values('Ny')
        central = sub.c_per_wall.to_numpy(dtype=float)
        low = pd.to_numeric(sub.ensemble_c_ci_low).to_numpy(dtype=float)
        high = pd.to_numeric(sub.ensemble_c_ci_high).to_numpy(dtype=float)
        yerr = np.vstack((np.maximum(0, central - low), np.maximum(0, high - central)))
        ax.errorbar(sub.Ny, central, yerr=yerr, color=color, marker=marker, ms=4, lw=1.0, ls=linestyle, capsize=2, label=label)
    ax.axhline(1.0, color='black', ls='--', lw=0.8, label='$c_w=1$' if panel == 0 else None)
    ax.set_title(rf'$n={order}$')
    ax.set_xlabel(r'$N_y$')
    ax.set_xticks(NY_VALUES)
    ax.tick_params(direction='in', top=True, right=True)
    ax.text(-0.18, 1.03, f'({chr(97+panel)})', transform=ax.transAxes, fontsize=9)
axes[0].set_ylabel(r'$c_{w,n}$')
axes[-1].legend(frameon=False, fontsize=6)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'renyi_central_charge_comparison.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'renyi_central_charge_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Cap, estimator-order, fit-window, and twist controls

This comparison makes the noncommutativity of the logarithm and ensemble average visible and exposes every preregistered sensitivity. The annealed covariance and flattened parent are controls, not substitutes for the quenched log-polar generator.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.05, 5.0))
ny = max(NY_VALUES)
ax = axes[0, 0]
for cap, color in zip(POLAR_LOG_CAPS, ('#d62728', '#2ca02c', '#1f77b4')):
    sub = fit_df[(fit_df.family == 'quenched_mean') & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    sub = sub[np.isclose(pd.to_numeric(sub.cap), cap) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    if len(sub):
        ax.plot(cap, sub.iloc[0].c_per_wall, marker='o', color=color)
ax.axhline(1.0, color='black', ls='--', lw=0.8)
ax.set_xlabel(r'polar cap $\Lambda$')
ax.set_ylabel(r'quenched $c_{w,1}$')
ax.set_title(rf'$N_y={ny}$ cap sensitivity')
ax.text(-0.14, 1.03, '(a)', transform=ax.transAxes, fontsize=9)
ax = axes[0, 1]
for family, color, marker, label in (
    ('quenched_mean', '#d62728', '^', 'quenched log'),
    ('annealed_covariance', '#9467bd', 's', 'annealed covariance'),
    ('flattened_parent', '#7f7f7f', 'o', 'flattened parent'),
):
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    if family == 'quenched_mean':
        sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    if len(sub):
        ax.plot([label], [sub.iloc[0].c_per_wall], color=color, marker=marker, ms=5, ls='none')
ax.axhline(1.0, color='black', ls='--', lw=0.8)
ax.set_ylabel(r'$c_{w,1}$')
ax.set_title('estimator-order controls')
ax.tick_params(axis='x', rotation=20)
ax.text(-0.14, 1.03, '(b)', transform=ax.transAxes, fontsize=9)
ax = axes[1, 0]
variants = ('primary', 'aymin6', 'drop_endpoint')
for fi, (family, (color, marker, _, label)) in enumerate(family_style.items()):
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1)]
    if family != 'actual_endpoint':
        sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    values = [sub[sub.fit_variant == variant].iloc[0].c_per_wall for variant in variants]
    ax.plot(np.arange(len(variants)) + 0.08 * (fi - 1), values, color=color, marker=marker, ms=4, lw=0.9, label=label)
ax.axhline(1.0, color='black', ls='--', lw=0.8)
ax.set_xticks(range(len(variants)), ('primary', r'$A_y^{\min}=6$', 'drop endpoint'), rotation=15)
ax.set_ylabel(r'$c_{w,1}$')
ax.set_title('fit-window sensitivity')
ax.text(-0.14, 1.03, '(c)', transform=ax.transAxes, fontsize=9)
ax = axes[1, 1]
for family, color, marker, label in (('quenched_mean', '#d62728', '^', 'quenched'), ('typical_sample', '#2ca02c', 's', 'typical')):
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP)].sort_values('twist')
    ax.plot(sub.twist * 1e7, sub.c_per_wall, color=color, marker=marker, ms=5, lw=0.9, label=label)
ax.axhline(1.0, color='black', ls='--', lw=0.8)
ax.set_xlabel(r'boundary twist $\phi\times10^7$')
ax.set_ylabel(r'$c_{w,1}$')
ax.set_title('degeneracy sensitivity')
ax.legend(frameon=False, fontsize=6)
ax.text(-0.14, 1.03, '(d)', transform=ax.transAxes, fontsize=9)
for ax in axes.flat:
    ax.tick_params(direction='in', top=True, right=True)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'cap_estimator_fit_twist_controls.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'cap_estimator_fit_twist_controls.png', dpi=300, bbox_inches='tight')
plt.show()

## Raw diagnostics and claim ledger

The final cell exposes the numerical maxima, censoring/cap counts, fitted rows, and machine-readable acceptance decisions. Failed cap stability is reported as endpoint non-identifiability rather than evidence against edge physics.

In [ ]:
display(fit_df.sort_values(['family', 'Ny', 'renyi_order', 'fit_variant']).reset_index(drop=True))
numeric_columns = [column for column in diagnostics_df.columns if column.endswith('residual') or column.endswith('coupling') or column.endswith('spread')]
display(diagnostics_df[numeric_columns].max(numeric_only=True).to_frame('maximum'))
display(diagnostics_df.groupby(['family', 'Ny', 'cap'], dropna=False).agg(rows=('Ny', 'size'), interior_modes=('interior_mode_count', 'median'), capped_modes=('capped_mode_count', 'median')).reset_index())
print(json.dumps(summary['claim_ledger'], indent=2))
print(f'analysis complete: {OUTPUT_ROOT}')